# Gemma Scope 2: one attribution graph

**Time Course MechInterp → Gemma 3 1B → official Scope 2 CLT → graph + pruning baseline.**
This notebook prepares data and generates a graph in the pinned `uv` environment.
It also saves circuit-tracer's native influence-pruning baseline.
Outputs go into `code/example_code/output/`.

In Colab: upload this notebook, choose **Runtime → Change runtime type → A100 GPU**
(if available) and high RAM, then run the cells in order. The first cell asks for
`cs795-colab.zip`, the small source bundle provided beside the notebook.
Accept the [Gemma model terms](https://huggingface.co/google/gemma-3-1b-pt) and add a
Colab secret named **HF_TOKEN**, enabling notebook access.

The CLT download is about **32.75 GB**. Allow at least **45 GB free disk**; roughly
**80–100 GB host RAM** is a planning estimate for the current loader, not a tested
minimum. One official-checkpoint prompt completed on an A100 80GB; peak host RAM
was not measured. Behavioral validation remains pending. Colab resource availability
varies even with paid compute units ([FAQ](https://research.google.com/colaboratory/faq.html)).
A laptop can run the data cells; graph generation requires a native-BF16 CUDA GPU.

### 1. Load the project
Locally, open this notebook inside the repository. In Colab, select only the supplied
`cs795-colab.zip` when prompted. No Git commit or public repository is required.

In [ ]:
# # gemma_scope2_baseline.ipynb
# Prepare prompts, generate a Scope 2 graph, and download the run artifacts.
#

from pathlib import Path
import io
import json
import os
import shutil
import subprocess
import sys
import uuid
import zipfile

# Reuse a local checkout; Colab receives the same source files through the ZIP.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "code/example_code/generate_graph.py").is_file()
             and (p / "code/example_code/prepare_data.py").is_file()
             and (p / "code/example_code/prune_graph.py").is_file()
             and (p / "pyproject.toml").is_file() and (p / "uv.lock").is_file()), None)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if ROOT is None:
    if not IN_COLAB:
        raise RuntimeError("Open this notebook from inside the project repository.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only cs795-colab.zip.")
    ROOT = Path("/content/cs795-project")
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as bundle:
        # Keep archive entries inside the project directory.
        if any(Path(n).is_absolute() or ".." in Path(n).parts for n in bundle.namelist()):
            raise ValueError("ZIP contains an unsafe path.")
        required = {"pyproject.toml", "uv.lock", ".python-version", "datasets/time_course.lock.json",
                    "code/example_code/prepare_data.py", "code/example_code/generate_graph.py",
                    "code/example_code/prune_graph.py"}
        if not required.issubset(bundle.namelist()):
            raise ValueError("Use the supplied cs795-colab.zip with repository-relative files.")
        bundle.extractall(ROOT)
assert (ROOT / "uv.lock").is_file(), "Use the supplied project ZIP."
os.chdir(ROOT)
OUT = ROOT / "code/example_code/output"
OUT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
print(f"Project: {ROOT}\nOutputs: {OUT}")

### 2. Install the locked environment
ML packages run in the project's Python environment, so Colab's preinstalled
package versions do not need to be replaced. This installs software, not weights.

In [ ]:
# Only uv is installed in the notebook kernel; the model stack lives in .venv.
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.10.2"],
                   check=True, cwd=ROOT, timeout=300)
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv was installed but is not on PATH; restart the notebook kernel.")
subprocess.run([UV, "sync", "--locked", "--extra", "graph"], check=True, cwd=ROOT, timeout=1200)
PY = [UV, "run", "--locked", "--extra", "graph", "python"]

### 3. Prepare and inspect the data
Each upstream JSON file describes a relation, prompt templates, and `subject` / `object`
pairs. We fill the template with the subject and keep the object as the expected answer.
Our two geography relations yield 33 examples; removing two answer-overlap cases leaves
**31 JSONL rows**, one prompt per line. Only `prompt_prefix` goes into Gemma.

In [ ]:
subprocess.run(PY + ["code/example_code/prepare_data.py"], check=True, cwd=ROOT, timeout=300)
prepared = ROOT / "datasets/prepared/time-course-mechinterp/prompts.jsonl"
shutil.copyfile(prepared, OUT / "prompts.jsonl")
rows = [json.loads(line) for line in prepared.read_text().splitlines()]
ROW = 0  # Change this index to inspect and trace another fact.
assert 0 <= ROW < len(rows)
print(f"{len(rows)} prepared prompts; selected row {ROW}")
print(json.dumps(rows[ROW], indent=2))

### 4. Check hardware and authenticate
This check runs before downloading the large checkpoints. Inspect the available RAM
and disk against the estimates above. A T4 does not support this example's native BF16.
Your Hugging Face token is read from Colab Secrets or a hidden local prompt.

In [ ]:
# Query the same Python/Torch environment that will generate the graph.
probe = """
import json, psutil, shutil, torch
print(json.dumps({
    "gpu": torch.cuda.get_device_name() if torch.cuda.is_available() else None,
    "available_host_RAM_GiB": round(psutil.virtual_memory().available / 2**30, 1),
    "free_disk_GiB": round(shutil.disk_usage('.').free / 2**30, 1),
}, indent=2), flush=True)
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported(including_emulation=False):
    raise SystemExit("Select a native-BF16 CUDA GPU runtime before continuing.")
"""
subprocess.run(PY + ["-c", probe], check=True, cwd=ROOT, timeout=60)
if not os.environ.get("HF_TOKEN"):
    if IN_COLAB:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    else:
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("Hugging Face token (hidden): ")
if not os.environ["HF_TOKEN"].strip():
    raise ValueError("A Hugging Face token with Gemma access is required.")
print("Authentication configured; token is not displayed.")

### 5. Generate one graph and save its pruning baseline
The first run downloads the pinned model and 26 CLT shards. Each attempt gets a new
output directory. Graph construction traces up to two likely next-token outputs,
with a cap of 128 feature nodes; the expected answer is not forced into the graph.

Edit the two thresholds below to set cumulative influence cutoffs for native pruning.
They control pruning after graph construction, and the same settings are used for the
viewer. They do not specify exact node or edge counts; the baseline records the achieved
sizes. Neither changing these thresholds nor lowering the feature cap reduces checkpoint
loading memory.


In [ ]:
NODE_THRESHOLD = 0.8  # Cumulative node-influence cutoff for native pruning.
EDGE_THRESHOLD = 0.98  # Cumulative edge-influence cutoff after node pruning.
RUN_DIR = OUT / f"graph-{uuid.uuid4().hex[:8]}"  # Preserve earlier attempts.
print(f"Saving this run to {RUN_DIR}")
try:
    subprocess.run(PY + ["code/example_code/generate_graph.py", "--row", str(ROW),
                        "--output", str(RUN_DIR), "--node-threshold", str(NODE_THRESHOLD),
                        "--edge-threshold", str(EDGE_THRESHOLD)],
                   check=True, cwd=ROOT, timeout=7200)
finally:
    # Display graph/baseline summaries and any saved failure receipt.
    if (RUN_DIR / "manifest.json").is_file():
        manifest = json.loads((RUN_DIR / "manifest.json").read_text())
        summary = {key: manifest[key] for key in
                   ("status", "stage", "error", "graph", "baseline") if key in manifest}
        print(json.dumps(summary, indent=2))


### 6. Save the results
Download the run before Colab disconnects. The ZIP contains this run's graph,
pruning baseline, viewer JSON, and manifest. Weights, environment files, and credentials
remain outside the archive.
If graph generation failed, this cell can still download the failure manifest.

In [ ]:
archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
else:
    print(f"Saved: {archive}")

Open the downloaded ZIP on your laptop. From the repository root, run this command,
replacing the example graph directory with the extracted run's name:

```sh
uv run --locked --extra graph circuit-tracer start-server \
  --graph_file_dir code/example_code/output/graph-REPLACE/viewer --port 8041
```

Open the local URL printed by the server. `graph.pt` preserves the graph before native
pruning, including its construction cap. `baseline/baseline.json` records the pruning
thresholds and achieved sizes; `baseline/pruning.pt` saves the native node/edge masks
for future comparisons. `viewer/` displays the graph pruned with those same thresholds.
`manifest.json` records sources, prompt, output probabilities, and run measurements.
Feature nodes identify a layer, token position, and feature index.

This is an attribution-only pruning baseline. Influence thresholds are cumulative
score cutoffs, not exact circuit sizes or percentages of behavioral faithfulness.
The saved masks support later comparisons; this notebook does not test interventions
or causal faithfulness.

Sources: [dataset](https://github.com/adhakimi/time-course-mechinterp/tree/0f018bbc8bd766cdf74930c445f0c9c4df9ac7bd/dataset/relations_data),
[official CLT](https://huggingface.co/google/gemma-scope-2-1b-pt/tree/b738dc06961818c011fb2e44a316352ca0f4e873/clt/width_262k_l0_medium_affine),
[native influence pruning](https://github.com/decoderesearch/circuit-tracer/blob/255d989e93be1299d53d67b59e9fd76a0e274b14/circuit_tracer/graph.py#L184-L313),
[attribution methods](https://transformer-circuits.pub/2025/attribution-graphs/methods.html).

In [1]:
import torch
import sys

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

Python: /scratch/njayotid/cs795-project-main/cs795-project-main/.venv-causal-circuit/bin/python
PyTorch: 2.8.0+cu128
CUDA available: True
GPU count: 2
GPU 0: NVIDIA A100-SXM4-80GB MIG 3g.40gb
GPU 1: NVIDIA A100-SXM4-80GB MIG 3g.40gb


In [2]:
from pathlib import Path

project_root = Path("/scratch/njayotid/cs795-project-main/cs795-project-main")

print("Project root exists:", project_root.exists())
print("Baseline notebook exists:", (project_root / "code/example_code/gemma_scope2_baseline.ipynb").exists())
print("Graph script exists:", (project_root / "code/example_code/generate_graph.py").exists())
print("Prepared prompts exist:", (project_root / "datasets/prepared/time-course-mechinterp/prompts.jsonl").exists())


Project root exists: True
Baseline notebook exists: True
Graph script exists: True
Prepared prompts exist: True


In [3]:
import json

prompts_path = project_root / "datasets/prepared/time-course-mechinterp/prompts.jsonl"

with open(prompts_path, "r", encoding="utf-8") as f:
    first_example = json.loads(next(f))

print(json.dumps(first_example, indent=2, ensure_ascii=False))

{
  "answer": "Brazil",
  "id": "semantic/city_in_country:0:0",
  "prompt_prefix": "Rio de Janeiro is part of the country of",
  "relation": "city_in_country",
  "source": {
    "path": "dataset/relations_data/semantic/city_in_country.json",
    "repository": "https://github.com/adhakimi/time-course-mechinterp",
    "revision": "0f018bbc8bd766cdf74930c445f0c9c4df9ac7bd",
    "sample_index": 0,
    "sha256": "bf2d79fa11a8c400240f5cf5da672164cbd1bfbc400718716b730bb8fbbc9dd5",
    "template_index": 0
  },
  "subject": "Rio de Janeiro"
}


In [4]:
from pathlib import Path

print("Project root:", project_root)

for folder_name in ["dataset", "datasets"]:
    folder = project_root / folder_name
    print(f"\n{folder_name} exists:", folder.exists())

    if folder.exists():
        matches = list(folder.rglob("city_in_country.json"))
        for path in matches:
            print("Found:", path)
            

Project root: /scratch/njayotid/cs795-project-main/cs795-project-main

dataset exists: False

datasets exists: True
Found: /scratch/njayotid/cs795-project-main/cs795-project-main/datasets/raw/time-course-mechinterp/factual/city_in_country.json
Found: /scratch/njayotid/cs795-project-main/cs795-project-main/datasets/raw/time-course-mechinterp/semantic/city_in_country.json
Found: /scratch/njayotid/cs795-project-main/cs795-project-main/datasets/relations_data/factual/city_in_country.json
Found: /scratch/njayotid/cs795-project-main/cs795-project-main/datasets/relations_data/semantic/city_in_country.json
